In [1]:
import os.path
import numpy as np
import pandas as pd
import re
from sklearn.model_selection import train_test_split
import os, time, json, math
from typing import List, Any, Tuple
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from fasterrisk.fasterrisk import RiskScoreOptimizer, RiskScoreClassifier
from sklearn.linear_model import LogisticRegression
from helpers import *

## Config

In [2]:
LABEL_COL = 'Cancer_lbl'
ID_COL    = 'pid'

FEATURES_STLMD  = ['sct_long_dia','part_solid','ground_glass','solid','Upper_Lobe','Spiculation','age','sex', 'sct_found_after_comp', 'sct_perp_dia', 'sct_ab_gwth']

# ≥ cutpoints for binning (age & size)
CUTS_CONT = {
    "age": [55, 58, 61, 64, 67],
    "sct_long_dia": [6, 8, 10, 15, 20, 30],
    "sct_perp_dia": [5, 7, 10, 12, 15, 20],
    "sct_ab_gwth": [2]
}

# binary passthroughs (copied as-is)
BIN_PASSTHROUGH_STLMD  = ['part_solid','ground_glass','solid','Upper_Lobe','Spiculation','sex', 'sct_found_after_comp']

# Data files
CSV1 = 'ml_dataset/nlst_ct_nodule_df_set1.csv'
CSV2 = 'ml_dataset/nlst_ct_nodule_df_set2.csv'

## Read Data

In [3]:
# -------------------------
# Load, normalize/encode
# -------------------------
df1 = pd.read_csv(CSV1)
df2 = pd.read_csv(CSV2)

df1 = filter_age_le_70(df1)
df2 = filter_age_le_70(df2)

df1 = normalize_and_encode(df1)
df2 = normalize_and_encode(df2)

df1.loc[df1['sct_ab_gwth'] == 9, 'sct_ab_gwth'] = 0
df2.loc[df2['sct_ab_gwth'] == 9, 'sct_ab_gwth'] = 0

df1['sct_ab_gwth'] = df1['sct_ab_gwth'].fillna(0)
df2['sct_ab_gwth'] = df2['sct_ab_gwth'].fillna(0)


df1 = df1.dropna(subset=['sct_perp_dia'])
df2 = df2.dropna(subset=['sct_perp_dia'])

# -------------------------
# Patient-level stratified split (on df1)
# -------------------------
patients = df1[[ID_COL, LABEL_COL]].drop_duplicates()
train_patients, val_patients = train_test_split(
    patients,
    test_size=0.2,
    stratify=patients[LABEL_COL],
    random_state=42
)
train_df = df1[df1[ID_COL].isin(train_patients[ID_COL])]
val_df   = df1[df1[ID_COL].isin(val_patients[ID_COL])]

## Subgroup analysis for Fasterrisk

In [4]:
def train_fasterrisk(
    train_df,
    val_df=None,
    sparsity_k=5,
    parent_size=50,
    gap_tolerance=0.15,
    select_top_m=100,
    max_attempts=200,
):
    """
    Train a FasterRisk model on the given train_df.

    Parameters
    ----------
    train_df : pd.DataFrame
        Training data (must contain FEATURES_STLMD and LABEL_COL columns).
    val_df : pd.DataFrame, optional
        Validation data. If None, uses the global val_df.
    sparsity_k, parent_size, gap_tolerance, select_top_m, max_attempts :
        FasterRisk hyperparameters.

    Returns
    -------
    multipliers : np.ndarray
    beta0_int   : np.ndarray
    betas_int   : np.ndarray
    """
    _val_df = val_df if val_df is not None else globals()["val_df"]

    # ---- prepare splits ----
    X_train_df, y_train_raw = prepare_data(train_df, FEATURES_STLMD, LABEL_COL)
    X_val_df,   y_val_raw   = prepare_data(_val_df,  FEATURES_STLMD, LABEL_COL)

    X_train_bin, X_val_bin, _ = binarize_and_align_custom(
        X_train_df, X_val_df, X_val_df,
        feature_cuts={
            "age":          CUTS_CONT["age"],
            "sct_long_dia": CUTS_CONT["sct_long_dia"],
            "sct_perp_dia": CUTS_CONT["sct_perp_dia"],
            "sct_ab_gwth":  CUTS_CONT["sct_ab_gwth"],
        },
        passthrough_binary=BIN_PASSTHROUGH_STLMD,
    )

    y_train = to_fastrisk_y(y_train_raw, pos_label=1)
    y_val   = to_fastrisk_y(y_val_raw,   pos_label=1)

    X_train = X_train_bin.to_numpy(dtype=float)
    X_val   = X_val_bin.to_numpy(dtype=float)

    # ---- optimize ----
    opt = make_optimizer(
        X=X_train, y=y_train,
        k=sparsity_k, parent_size=parent_size,
        gap_tolerance=gap_tolerance,
        select_top_m=select_top_m,
        max_attempts=max_attempts,
        want_intercept=True,
    )
    t0 = time.time()
    opt.optimize()
    print("Optimization takes {:.2f} seconds.".format(time.time() - t0))

    multipliers, beta0_int, betas_int = extract_models(opt.get_models())
    print("Generated {} risk score models from the sparse diverse pool.".format(len(multipliers)))

    return multipliers, beta0_int, betas_int


In [5]:
train_df1 = train_df[train_df['study_yr'] == 0].copy()
train_df2 = train_df[train_df['study_yr'] != 0].copy()
multipliers, beta0_int, betas_int = train_fasterrisk(train_df)
multipliers1, beta0_int1, betas_int1 = train_fasterrisk(train_df1)
multipliers2, beta0_int2, betas_int2 = train_fasterrisk(train_df2)

Optimization takes 124.34 seconds.
Generated 82 risk score models from the sparse diverse pool.
Optimization takes 23.85 seconds.
Generated 76 risk score models from the sparse diverse pool.
Optimization takes 93.86 seconds.
Generated 74 risk score models from the sparse diverse pool.


In [ ]:
def compute_auc_with_ci(
    multipliers, beta0_int, betas_int,
    test_data_dict: dict,
    train_df_for_align: pd.DataFrame,
    ci=0.95,
    n_bootstrap=1000,
    random_state=42,
):
    """
    Compute the AUC and its confidence interval for the FasterRisk model on multiple test datasets.

    Parameters:
        multipliers: np.ndarray, shape (n_models,), multiplier for each model
        beta0_int: np.ndarray, shape (n_models,), intercept for each model
        betas_int: np.ndarray, shape (n_models, n_features), coefficients for each model
        test_data_dict: dict, keys are dataset names, values are test DataFrames
        train_df_for_align: pd.DataFrame, used to construct binary feature columns consistent with training
        ci: float, confidence level, default 0.95 (i.e., 95%)
        n_bootstrap: int, number of bootstrap iterations, default 1000
        random_state: int, random seed, default 42

    Returns:
        pd.DataFrame, containing the following columns:
            - dataset: str, dataset name
            - auc_mean: float, mean AUC
            - auc_ci_lower: float, lower bound of the AUC confidence interval
            - auc_ci_upper: float, upper bound of the AUC confidence interval
            - n_samples: int, number of samples
            - n_models: int, number of models in the ensemble
    """
    results = []
    alpha = 1 - ci
    rng = np.random.default_rng(random_state)

    X_train_align_df, _ = prepare_data(train_df_for_align, FEATURES_STLMD, LABEL_COL)

    for dataset_name, test_df in test_data_dict.items():
        # Generate X_test and y_test from the test DataFrame
        X_test_df, y_test_raw = prepare_data(test_df, FEATURES_STLMD, LABEL_COL)

        _, _, X_test_bin = binarize_and_align_custom(
            X_train_align_df, X_test_df, X_test_df,
            feature_cuts={
                "age": CUTS_CONT["age"],
                "sct_long_dia": CUTS_CONT["sct_long_dia"],
                "sct_perp_dia": CUTS_CONT["sct_perp_dia"],
                "sct_ab_gwth": CUTS_CONT["sct_ab_gwth"],
            },
            passthrough_binary=BIN_PASSTHROUGH_STLMD,
        )

        y_test = to_fastrisk_y(y_test_raw, pos_label=1)
        y_test_01 = (y_test + 1.0) / 2.0
        X_test = X_test_bin.to_numpy(dtype=float)
        n_samples = X_test.shape[0]

        # Compute predicted probabilities for all models
        n_models = len(multipliers)
        prob_matrix = np.zeros((n_samples, n_models))

        for i in range(n_models):
            mult = float(multipliers[i])
            b0 = float(beta0_int[i])
            betas = np.asarray(betas_int[i], dtype=int)
            probs = model_probs(mult, b0, betas, X_test)
            prob_matrix[:, i] = probs

        # Ensemble prediction (average probability across all models)
        ensemble_probs = prob_matrix.mean(axis=1)

        # AUC
        fpr, tpr, _ = roc_curve(y_test_01, ensemble_probs)
        auc_original = auc(fpr, tpr)

        # Bootstrap CI
        bootstrap_aucs = []
        for _ in range(n_bootstrap):
            indices = rng.choice(n_samples, size=n_samples, replace=True)
            y_boot = y_test_01[indices]
            probs_boot = ensemble_probs[indices]

            if np.unique(y_boot).size < 2:
                continue

            fpr_boot, tpr_boot, _ = roc_curve(y_boot, probs_boot)
            auc_boot = auc(fpr_boot, tpr_boot)
            bootstrap_aucs.append(auc_boot)

        if len(bootstrap_aucs) == 0:
            ci_lower, ci_upper = np.nan, np.nan
        else:
            bootstrap_aucs = np.array(bootstrap_aucs)
            lower_percentile = (alpha / 2) * 100
            upper_percentile = (1 - alpha / 2) * 100
            ci_lower = np.percentile(bootstrap_aucs, lower_percentile)
            ci_upper = np.percentile(bootstrap_aucs, upper_percentile)

        results.append({
            "dataset": dataset_name,
            "auc_mean": float(auc_original),
            "auc_ci_lower": float(ci_lower) if not np.isnan(ci_lower) else np.nan,
            "auc_ci_upper": float(ci_upper) if not np.isnan(ci_upper) else np.nan,
        })

    return pd.DataFrame(results)

In [7]:
def transform_auc_df(df, ci_col_name):
    """
    Convert a 4-column dataframe into 2 columns:
    - keep the first column unchanged (e.g., dataset)
    - merge the next three columns into one CI string column
    """
    if df.shape[1] != 4:
        raise ValueError("Input dataframe must have exactly 4 columns.")

    first_col = df.columns[0]
    mean_col, lower_col, upper_col = df.columns[1:4]

    def _fmt(x):
        return f"{x:.2f}" if pd.notna(x) else "NA"

    out_df = df[[first_col]].copy()
    out_df[ci_col_name] = (
        df[mean_col].map(_fmt)
        + " ("
        + df[lower_col].map(_fmt)
        + ", "
        + df[upper_col].map(_fmt)
        + ")"
    )
    return out_df

In [8]:
test_data_dict = {
    "all": df2,
    "1st_scr": df2[df2["study_yr"] == 0].copy(),
    "rest_scr": df2[df2["study_yr"] != 0].copy(),
}

auc_ci_df = compute_auc_with_ci(
    multipliers, beta0_int, betas_int,
    test_data_dict=test_data_dict,
    train_df_for_align=train_df,
    ci=0.95,
    n_bootstrap=1000,
 )

In [9]:
auc_ci_df1 = compute_auc_with_ci(
    multipliers1, beta0_int1, betas_int1,
    test_data_dict=test_data_dict,
    train_df_for_align=train_df1,
    ci=0.95,
    n_bootstrap=1000,
)

In [10]:
auc_ci_df2 = compute_auc_with_ci(
    multipliers2, beta0_int2, betas_int2,
    test_data_dict=test_data_dict,
    train_df_for_align=train_df2,
    ci=0.95,
    n_bootstrap=1000,
)

In [23]:
fstrsk_all = transform_auc_df(auc_ci_df, "fstrsk_all")
fstrsk_1st = transform_auc_df(auc_ci_df1, "fstrsk_1st")
fstrsk_rest = transform_auc_df(auc_ci_df2, "fstrsk_rest")

fstrsk_merged = fstrsk_all.merge(fstrsk_1st, on="dataset").merge(fstrsk_rest, on="dataset")
fstrsk_merged.style.hide(axis="index").set_caption("FasterRisk").set_properties(**{"text-align":"center","padding":"6px","background-color":"#050D14"}).set_table_styles([{"selector":"th","props":[("background-color","#1f2937"),("color","white"),("text-align","center")]}])

dataset,fstrsk_all,fstrsk_1st,fstrsk_rest
all,"0.76 (0.75, 0.78)","0.68 (0.66, 0.70)","0.75 (0.73, 0.77)"
1st_scr,"0.81 (0.77, 0.84)","0.80 (0.76, 0.83)","0.80 (0.76, 0.83)"
rest_scr,"0.75 (0.73, 0.77)","0.66 (0.63, 0.68)","0.74 (0.72, 0.76)"


## Subgroup analysis for PanCan

In [12]:
# coefficients of pancan
pancan_coef = {
    'age': 0.0286687,
    'gender': 0.6010727,
    'Family_History': 0.29611,
    'diagemph': 0.2953112,
    'sct_long_dia': -5.385484,
    'ground_glass': -0.1276173,
    'part_solid': 0.3769578,
    'Upper_Lobe': 0.6581383,
    'Spiculation': 0.7729335,
    'num_nodule': -0.0824156,
    'intercept': -6.78917
}

df2_pancan = df2.copy()
df2_pancan['sct_long_dia'] = df2_pancan['sct_long_dia']-4
df2_pancan['age'] = df2_pancan['age']-62
df2_pancan['num_nodule'] = df2_pancan['num_nodule']-4
df2_pancan['sct_long_dia'] = (df2_pancan['sct_long_dia'] / 10) ** (-0.5) - 1.58113883
df2_pancan['gender'] = df2_pancan['gender'].map({'male': 0, 'female': 1}).astype(int)

test_data_dict_pancan = {
    "all": df2_pancan,
    "1st_scr": df2_pancan[df2_pancan["study_yr"] == 0].copy(),
    "rest_scr": df2_pancan[df2_pancan["study_yr"] != 0].copy(),
}


def pancan_predict_proba(df: pd.DataFrame, coef: dict) -> np.ndarray:
    intercept = coef['intercept']
    X = df.reindex(columns=list(coef.keys())).copy()
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0.0)
    linear_term = intercept + X.mul(pd.Series(coef), axis=1).sum(axis=1)
    return 1.0 / (1.0 + np.exp(-linear_term.to_numpy(dtype=float)))

In [13]:
# PanCan: AUC + 95% CI on three preprocessed test sets
alpha = 0.05
n_bootstrap = 1000
rng = np.random.default_rng(42)

pancan_results = []

for dataset_name, df_test in test_data_dict_pancan.items():
    y_true = pd.to_numeric(df_test[LABEL_COL], errors='coerce')
    y_score = pancan_predict_proba(df_test, pancan_coef)

    valid_mask = y_true.notna()
    y_true = y_true[valid_mask].to_numpy(dtype=int)
    y_score = y_score[valid_mask.to_numpy()]

    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc_original = auc(fpr, tpr)

    bootstrap_aucs = []
    n_samples = len(y_true)
    for _ in range(n_bootstrap):
        idx = rng.choice(n_samples, size=n_samples, replace=True)
        y_boot = y_true[idx]
        s_boot = y_score[idx]

        if np.unique(y_boot).size < 2:
            continue

        fpr_b, tpr_b, _ = roc_curve(y_boot, s_boot)
        bootstrap_aucs.append(auc(fpr_b, tpr_b))

    if len(bootstrap_aucs) == 0:
        ci_lower, ci_upper = np.nan, np.nan
    else:
        ci_lower = np.percentile(bootstrap_aucs, (alpha / 2) * 100)
        ci_upper = np.percentile(bootstrap_aucs, (1 - alpha / 2) * 100)

    pancan_results.append({
        'dataset': dataset_name,
        'auc_mean': float(auc_original),
        'auc_ci_lower': float(ci_lower) if not np.isnan(ci_lower) else np.nan,
        'auc_ci_upper': float(ci_upper) if not np.isnan(ci_upper) else np.nan,
    })

pancan_auc_ci_df = pd.DataFrame(pancan_results)

In [24]:
pancan_auc_ci_df

,dataset,auc_mean,auc_ci_lower,auc_ci_upper
0,all,0.712334,0.695453,0.730988
1,1st_scr,0.799047,0.760414,0.833624
2,rest_scr,0.698480,0.679789,0.716431


## Subgroup analysis for LungRads

In [14]:
def lungrads_predict(row) -> str:
    lungrads = "1"
    diameter = pd.to_numeric(row.get('sct_long_dia', np.nan), errors='coerce')
    if pd.isna(diameter):
        return lungrads

    nodule_type = 'unknown'
    if int(pd.to_numeric(row.get('solid', 0), errors='coerce') or 0) == 1:
        nodule_type = 'solid'
    elif int(pd.to_numeric(row.get('part_solid', 0), errors='coerce') or 0) == 1:
        nodule_type = 'part-solid'
    elif int(pd.to_numeric(row.get('ground_glass', 0), errors='coerce') or 0) == 1:
        nodule_type = 'ground-glass'

    solid_component = pd.to_numeric(row.get('sct_perp_dia', np.nan), errors='coerce')

    airway_text = str(
        row.get('airway_level', row.get('airway_location', row.get('sct_epi_loc', '')))
    ).lower()
    airway_nodule_flag = int(pd.to_numeric(row.get('airway_nodule', row.get('airway', 0)), errors='coerce') or 0) == 1
    airway_segmental_or_proximal = airway_nodule_flag and any(
        key in airway_text for key in ['segment', 'proximal', 'trachea', 'main', 'lobar']
    )

    thick_walled_cyst = int(pd.to_numeric(row.get('thick_walled_cyst', 0), errors='coerce') or 0) == 1
    multilocular_cyst = int(pd.to_numeric(row.get('multilocular_cyst', 0), errors='coerce') or 0) == 1

    if nodule_type == 'solid' and diameter >= 15:
        lungrads = "4B"
    elif nodule_type == 'part-solid' and (pd.notna(solid_component) and solid_component >= 8):
        lungrads = "4B"
    elif nodule_type == 'solid' and 8 <= diameter < 15:
        lungrads = "4A"
    elif nodule_type == 'part-solid' and diameter >= 6 and (pd.notna(solid_component) and 6 <= solid_component < 8):
        lungrads = "4A"
    elif airway_segmental_or_proximal:
        lungrads = "4A"
    elif thick_walled_cyst or multilocular_cyst:
        lungrads = "4A"
    elif nodule_type == 'solid' and 6 <= diameter < 8:
        lungrads = "3"
    elif nodule_type == 'part-solid' and diameter >= 6 and (pd.isna(solid_component) or solid_component < 6):
        lungrads = "3"
    elif nodule_type == 'ground-glass' and diameter >= 30:
        lungrads = "3"
    elif nodule_type == 'solid' and diameter < 6:
        lungrads = "2"
    elif nodule_type == 'part-solid' and diameter < 6:
        lungrads = "2"
    elif nodule_type == 'ground-glass' and diameter < 30:
        lungrads = "2"

    suspicious_margins = 'Spicul' in str(row.get('sct_margins', '')).lower()
    suspicious_spiculation = int(pd.to_numeric(row.get('Spiculation', 0), errors='coerce') or 0) == 1
    if lungrads in ['3', '4A', '4B'] and (suspicious_margins or suspicious_spiculation):
        lungrads = '4X'

    return lungrads

In [15]:
# df2["LungRADS"] = df2.apply(lungrads_predict, axis=1)

## Subgroup Analysis for PanCan_NLST

In [16]:
test_data_dict_pancan = {
    "all": df2_pancan,
    "1st_scr": df2_pancan[df2_pancan["study_yr"] == 0].copy(),
    "rest_scr": df2_pancan[df2_pancan["study_yr"] != 0].copy(),
}

In [17]:
def fit_pancan_coef_from_train_df(
    train_df: pd.DataFrame,
    target_col: str = "Cancer_lbl",
    feature_cols=None,
    C: float = 1.0,
    max_iter: int = 200000,
) -> dict:

    if feature_cols is None:
        feature_cols = [
            "age",
            "gender",
            "Family_History",
            "diagemph",
            "sct_long_dia",
            "ground_glass",
            "part_solid",
            "Upper_Lobe",
            "Spiculation",
            "num_nodule",
        ]

    df_fit = train_df.copy()

    # PanCan-style transforms used in your notebook.
    df_fit["age"] = pd.to_numeric(df_fit["age"], errors="coerce") - 62
    df_fit["num_nodule"] = pd.to_numeric(df_fit["num_nodule"], errors="coerce") - 4

    dia = pd.to_numeric(df_fit["sct_long_dia"], errors="coerce") - 4
    dia = dia.where(dia > 0, np.nan)
    df_fit["sct_long_dia"] = (dia / 10.0) ** (-0.5) - 1.58113883

    gender_map = {"male": 0, "female": 1}
    if "gender" in df_fit.columns:
        df_fit["gender"] = (
            df_fit["gender"]
            .astype(str)
            .str.strip()
            .str.lower()
            .map(gender_map)
            .where(lambda s: s.notna(), pd.to_numeric(df_fit["gender"], errors="coerce"))
        )

    X = df_fit[feature_cols].copy()
    for col in feature_cols:
        X[col] = pd.to_numeric(X[col], errors="coerce").fillna(0.0)

    y = pd.to_numeric(df_fit[target_col], errors="coerce")
    if set(pd.Series(y.dropna().unique()).astype(int).tolist()) == {-1, 1}:
        y = ((y + 1) / 2).astype(int)
    else:
        y = y.fillna(0).astype(int)

    lr = LogisticRegression(C=C, max_iter=max_iter, solver="lbfgs")
    lr.fit(X, y)

    coef_dict = {name: float(weight) for name, weight in zip(feature_cols, lr.coef_[0])}
    coef_dict["intercept"] = float(lr.intercept_[0])

    return coef_dict

In [18]:
pancan_coef_all = fit_pancan_coef_from_train_df(train_df)
pancan_coef_study0 = fit_pancan_coef_from_train_df(train_df1)
pancan_coef_study_non0 = fit_pancan_coef_from_train_df(train_df2)

pancan_models = {
    "all": pancan_coef_all,
    "study_yr_0": pancan_coef_study0,
    "study_yr_non0": pancan_coef_study_non0,
}

In [19]:
# PanCan(pancan_coef_all): AUC + 95% CI on three preprocessed test sets
alpha = 0.05
n_bootstrap = 1000
rng = np.random.default_rng(42)

pancan_results_all = []

for dataset_name, df_test in test_data_dict_pancan.items():
    y_true = pd.to_numeric(df_test[LABEL_COL], errors='coerce')
    y_score = pancan_predict_proba(df_test, pancan_coef_all)

    valid_mask = y_true.notna()
    y_true = y_true[valid_mask].to_numpy(dtype=int)
    y_score = y_score[valid_mask.to_numpy()]

    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc_original = auc(fpr, tpr)

    bootstrap_aucs = []
    n_samples = len(y_true)
    for _ in range(n_bootstrap):
        idx = rng.choice(n_samples, size=n_samples, replace=True)
        y_boot = y_true[idx]
        s_boot = y_score[idx]

        if np.unique(y_boot).size < 2:
            continue

        fpr_b, tpr_b, _ = roc_curve(y_boot, s_boot)
        bootstrap_aucs.append(auc(fpr_b, tpr_b))

    if len(bootstrap_aucs) == 0:
        ci_lower, ci_upper = np.nan, np.nan
    else:
        ci_lower = np.percentile(bootstrap_aucs, (alpha / 2) * 100)
        ci_upper = np.percentile(bootstrap_aucs, (1 - alpha / 2) * 100)

    pancan_results_all.append({
        'dataset': dataset_name,
        'auc_mean': float(auc_original),
        'auc_ci_lower': float(ci_lower) if not np.isnan(ci_lower) else np.nan,
        'auc_ci_upper': float(ci_upper) if not np.isnan(ci_upper) else np.nan,
    })

pancan_auc_ci_df_all = pd.DataFrame(pancan_results_all)

In [20]:
# PanCan(pancan_coef_all): AUC + 95% CI on three preprocessed test sets
alpha = 0.05
n_bootstrap = 1000
rng = np.random.default_rng(42)

pancan_results_1 = []

for dataset_name, df_test in test_data_dict_pancan.items():
    y_true = pd.to_numeric(df_test[LABEL_COL], errors='coerce')
    y_score = pancan_predict_proba(df_test, pancan_coef_study0)

    valid_mask = y_true.notna()
    y_true = y_true[valid_mask].to_numpy(dtype=int)
    y_score = y_score[valid_mask.to_numpy()]

    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc_original = auc(fpr, tpr)

    bootstrap_aucs = []
    n_samples = len(y_true)
    for _ in range(n_bootstrap):
        idx = rng.choice(n_samples, size=n_samples, replace=True)
        y_boot = y_true[idx]
        s_boot = y_score[idx]

        if np.unique(y_boot).size < 2:
            continue

        fpr_b, tpr_b, _ = roc_curve(y_boot, s_boot)
        bootstrap_aucs.append(auc(fpr_b, tpr_b))

    if len(bootstrap_aucs) == 0:
        ci_lower, ci_upper = np.nan, np.nan
    else:
        ci_lower = np.percentile(bootstrap_aucs, (alpha / 2) * 100)
        ci_upper = np.percentile(bootstrap_aucs, (1 - alpha / 2) * 100)

    pancan_results_1.append({
        'dataset': dataset_name,
        'auc_mean': float(auc_original),
        'auc_ci_lower': float(ci_lower) if not np.isnan(ci_lower) else np.nan,
        'auc_ci_upper': float(ci_upper) if not np.isnan(ci_upper) else np.nan,
    })

pancan_auc_ci_df_1 = pd.DataFrame(pancan_results_1)

In [21]:
# PanCan(pancan_coef_all): AUC + 95% CI on three preprocessed test sets
alpha = 0.05
n_bootstrap = 1000
rng = np.random.default_rng(42)

pancan_results_2 = []

for dataset_name, df_test in test_data_dict_pancan.items():
    y_true = pd.to_numeric(df_test[LABEL_COL], errors='coerce')
    y_score = pancan_predict_proba(df_test, pancan_coef_study_non0)

    valid_mask = y_true.notna()
    y_true = y_true[valid_mask].to_numpy(dtype=int)
    y_score = y_score[valid_mask.to_numpy()]

    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc_original = auc(fpr, tpr)

    bootstrap_aucs = []
    n_samples = len(y_true)
    for _ in range(n_bootstrap):
        idx = rng.choice(n_samples, size=n_samples, replace=True)
        y_boot = y_true[idx]
        s_boot = y_score[idx]

        if np.unique(y_boot).size < 2:
            continue

        fpr_b, tpr_b, _ = roc_curve(y_boot, s_boot)
        bootstrap_aucs.append(auc(fpr_b, tpr_b))

    if len(bootstrap_aucs) == 0:
        ci_lower, ci_upper = np.nan, np.nan
    else:
        ci_lower = np.percentile(bootstrap_aucs, (alpha / 2) * 100)
        ci_upper = np.percentile(bootstrap_aucs, (1 - alpha / 2) * 100)

    pancan_results_2.append({
        'dataset': dataset_name,
        'auc_mean': float(auc_original),
        'auc_ci_lower': float(ci_lower) if not np.isnan(ci_lower) else np.nan,
        'auc_ci_upper': float(ci_upper) if not np.isnan(ci_upper) else np.nan,
    })

pancan_auc_ci_df_2 = pd.DataFrame(pancan_results_2)

In [22]:
pancan_all = transform_auc_df(pancan_auc_ci_df_all, 'pancan_all')
pancan_1st = transform_auc_df(pancan_auc_ci_df_1, 'pancan_1st')
pancan_rest = transform_auc_df(pancan_auc_ci_df_2, 'pancan_rest')

pancan_merged = pancan_all.merge(pancan_1st, on="dataset").merge(pancan_rest, on="dataset")
pancan_merged.style.hide(axis="index").set_caption("Pancan NLST").set_properties(**{"text-align":"center","padding":"6px","background-color":"#050D14"}).set_table_styles([{"selector":"th","props":[("background-color","#1f2937"),("color","white"),("text-align","center")]}])

dataset,pancan_all,pancan_1st,pancan_rest
all,"0.72 (0.70, 0.74)","0.70 (0.68, 0.72)","0.71 (0.69, 0.73)"
1st_scr,"0.79 (0.76, 0.83)","0.80 (0.76, 0.83)","0.78 (0.74, 0.81)"
rest_scr,"0.71 (0.69, 0.72)","0.69 (0.67, 0.70)","0.70 (0.68, 0.72)"
